# Frozen GPT‑2 baseline evaluation

Воспроизводимая offline-оценка исходной next-item модели до изменения политики.

In [ ]:
# Выполните один раз в новом окружении (Colab/Jupyter). Затем при необходимости перезапустите kernel.
%pip install -q "torch>=2.6,<3" "transformers==4.57.1" "numpy>=1.26,<3" "pandas>=2.2,<4" "tqdm>=4.66,<5" safetensors


In [ ]:
from pathlib import Path

# Colab: загрузите prepared.zip и веса в /content или подключите Google Drive.
# Локально можно использовать Path(r"C:\...\prepared.zip").
DATA_PATH = Path("prepared.zip")        # либо Path("prepared")
CHECKPOINT_PATH = Path("best_checkpoint")  # папка save_pretrained ИЛИ best.pt / best.pth
SPLIT = "test"                         # "validation" для проверок во время RL
BATCH_SIZE = 32                        # при нехватке памяти уменьшите
OUTPUT_DIR = Path("gpt2_baseline_test") # новая папка; существующая не перезаписывается
REFERENCE_DIR = None                   # после RL: Path("gpt2_baseline_test")
# Для весов .pt/.pth архитектура берётся из prepared/resolved_config.json.
# Если обучали другую архитектуру, укажите её config.json:
MODEL_CONFIG_JSON = None
STATE_DICT_KEY = None                  # обычно распознаются model_state_dict/state_dict/model


In [ ]:
import os, sys, json, hashlib, zipfile, platform, importlib.metadata, time
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import torch
from transformers import GPT2Config, GPT2LMHeadModel
from tqdm.auto import tqdm
from IPython.display import display

assert SPLIT in {"test", "validation"}
assert BATCH_SIZE > 0
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
torch.manual_seed(42)
np.random.seed(42)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

environment = {"python": sys.version, "platform": platform.platform(), "device": str(DEVICE),
               "cuda": torch.version.cuda,
               "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
               "packages": {k: importlib.metadata.version(k) for k in
                            ["torch", "transformers", "numpy", "pandas", "safetensors"]}}
print(json.dumps(environment, indent=2, ensure_ascii=False))


In [ ]:
# Читаем только ожидаемые файлы, не выполняем код из архива.
required = [f"{SPLIT}_examples.npz", "item_mapping.json", "resolved_config.json",
            "data_statistics.json", "split_boundaries.json", "train_interactions.csv"]
if SPLIT == "test":
    required.append("validation_interactions.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Укажите DATA_PATH: архив/папка не найдены: {DATA_PATH.resolve()}")
if DATA_PATH.is_file():
    assert zipfile.is_zipfile(DATA_PATH), "DATA_PATH должен быть ZIP или папкой"
    # Отдельная папка по хешу исключает смешивание разных архивов.
    DATA_DIR = Path("gpt2_eval_data") / sha256(DATA_PATH)[:16]
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATA_PATH) as z:
        for name in required:
            matches = [n for n in z.namelist() if n.rsplit("/", 1)[-1] == name
                       and not n.startswith("__MACOSX/")]
            if len(matches) != 1:
                raise ValueError(f"Нужен ровно один {name}, найдено: {matches}")
            with z.open(matches[0]) as src, open(DATA_DIR / name, "wb") as dst:
                import shutil
                shutil.copyfileobj(src, dst)
else:
    DATA_DIR = DATA_PATH
    if not (DATA_DIR / "resolved_config.json").exists() and (DATA_DIR / "prepared").is_dir():
        DATA_DIR = DATA_DIR / "prepared"
for name in required:
    if not (DATA_DIR / name).is_file():
        raise FileNotFoundError(DATA_DIR / name)

cfg = read_json(DATA_DIR / "resolved_config.json")
mapping = read_json(DATA_DIR / "item_mapping.json")
stats = read_json(DATA_DIR / "data_statistics.json")
boundaries = read_json(DATA_DIR / "split_boundaries.json")
item_to_index = mapping["item_id_to_index"]
N_ITEMS = len(item_to_index)
PAD = int(mapping["pad_token_id"])
assert sorted(map(int, item_to_index.values())) == list(range(N_ITEMS))
assert PAD == N_ITEMS and N_ITEMS >= 20
index_to_item = np.empty(N_ITEMS, dtype=np.int64)
for item_id, index in item_to_index.items():
    index_to_item[int(index)] = int(item_id)

with np.load(DATA_DIR / f"{SPLIT}_examples.npz", allow_pickle=False) as z:
    users, histories, lengths, targets = [z[k].copy() for k in ["users", "histories", "lengths", "targets"]]
N = len(users)
assert N > 0 and histories.ndim == 2 and len(histories) == N
assert lengths.shape == targets.shape == (N,) and len(np.unique(users)) == N
assert np.all((lengths >= 1) & (lengths <= histories.shape[1]))
mask = np.arange(histories.shape[1])[None, :] < lengths[:, None]
assert np.all((histories[mask] >= 0) & (histories[mask] < N_ITEMS))
assert np.all(histories[~mask] == PAD)
assert np.all((targets >= 0) & (targets < N_ITEMS))
assert N == stats["splits"][SPLIT]["evaluation"]["eligible_users"]
del mask
print(f"{SPLIT}: {N:,} пользователей; каталог {N_ITEMS:,}; max context {histories.shape[1]}")
display(pd.DataFrame(boundaries).T)
display(pd.Series(stats["splits"][SPLIT]["evaluation"], name="Аудит когорты"))


In [ ]:
# Полная история просмотров. Код пары = номер пользователя в NPZ * N_ITEMS + индекс фильма.
# Сортированный массив занимает меньше памяти, чем множество Python для каждого пользователя.
user_index = pd.Index(users)
history_sources = ["train_interactions.csv"]
if SPLIT == "test":
    history_sources.append("validation_interactions.csv")
parts = []
for name in history_sources:
    for chunk in pd.read_csv(DATA_DIR / name, usecols=["user_id", "item_id", "last_watch_dt"],
                             dtype={"user_id":"int64", "item_id":"int64", "last_watch_dt":"string"},
                             chunksize=300_000):
        dates = pd.to_datetime(chunk["last_watch_dt"], format="%Y-%m-%d", errors="raise")
        # Все источники должны завершаться до начала оцениваемого периода.
        assert dates.notna().all() and (dates < pd.Timestamp(boundaries[SPLIT]["start"])).all()
        u = user_index.get_indexer(chunk["user_id"])
        v = chunk["item_id"].astype(str).map(item_to_index)
        good = (u >= 0) & v.notna().to_numpy()
        parts.append(u[good].astype(np.int64) * N_ITEMS + v[good].to_numpy(dtype=np.int64))
seen_pairs = np.unique(np.concatenate(parts))
del parts, chunk

def contains_sorted(sorted_values, queries):
    queries = np.asarray(queries)
    if len(sorted_values) == 0:
        return np.zeros(queries.shape, dtype=bool)
    pos = np.searchsorted(sorted_values, queries)
    return (pos < len(sorted_values)) & (sorted_values[np.minimum(pos, len(sorted_values)-1)] == queries)

def seen_flags(recommendations, rows):
    return contains_sorted(seen_pairs, np.asarray(rows)[:, None] * N_ITEMS + recommendations)

# Проверка: каждый фильм модельного контекста должен быть в полной истории того же пользователя.
for start in range(0, N, 10_000):
    rows = np.arange(start, min(start+10_000, N))
    valid = np.arange(histories.shape[1])[None, :] < lengths[rows, None]
    assert seen_flags(histories[rows], rows)[valid].all(), "NPZ и CSV истории не совпадают"
target_seen = contains_sorted(seen_pairs, np.arange(N, dtype=np.int64) * N_ITEMS + targets)
print(f"Известных пар пользователь–фильм в истории: {len(seen_pairs):,}")
print(f"Доля targets, уже встречавшихся пользователю: {target_seen.mean():.4%}")


In [ ]:
def load_checkpoint(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Укажите CHECKPOINT_PATH с обученными весами: {path.resolve()}")
    if path.is_dir():
        # Папка, сохранённая model.save_pretrained(...), со своим config.json.
        model, loading_info = GPT2LMHeadModel.from_pretrained(
            str(path), local_files_only=True, attn_implementation="eager", output_loading_info=True)
        problems = {k:v for k,v in loading_info.items() if v}
        if problems:
            raise ValueError(f"Checkpoint загружен не полностью: {problems}")
    else:
        if MODEL_CONFIG_JSON is not None:
            model_cfg = GPT2Config.from_json_file(str(MODEL_CONFIG_JSON))
        else:
            valid_keys = set(GPT2Config().to_dict())
            kwargs = {k:v for k,v in cfg["model"].items() if k in valid_keys}
            model_cfg = GPT2Config(**kwargs)
        model_cfg._attn_implementation = "eager"
        model = GPT2LMHeadModel(model_cfg)
        if path.suffix == ".safetensors":
            from safetensors.torch import load_file
            state = load_file(str(path), device="cpu")
        else:
            state = torch.load(path, map_location="cpu", weights_only=True)
        if STATE_DICT_KEY is not None:
            state = state[STATE_DICT_KEY]
        elif isinstance(state, dict):
            for key in ["model_state_dict", "state_dict", "model"]:
                if key in state and isinstance(state[key], dict):
                    state = state[key]
                    break
        if not isinstance(state, dict) or not all(isinstance(v, torch.Tensor) for v in state.values()):
            raise ValueError("Не найден state_dict. Укажите STATE_DICT_KEY или папку save_pretrained.")
        for prefix in ["module.", "_orig_mod."]:
            if state and all(k.startswith(prefix) for k in state):
                state = {k[len(prefix):]:v for k,v in state.items()}
        # Safetensors может хранить лишь одну копию связанных весов.
        if model.config.tie_word_embeddings:
            a, b = "transformer.wte.weight", "lm_head.weight"
            if a in state and b not in state: state[b] = state[a]
            if b in state and a not in state: state[a] = state[b]
            if a in state and b in state and not torch.equal(state[a], state[b]):
                raise ValueError("Связанные embedding/lm_head имеют разные веса; проверьте config.")
        model.load_state_dict(state, strict=True)
    assert model.config.vocab_size == N_ITEMS + 1, "Словарь весов не совпадает с prepared"
    assert model.config.pad_token_id == PAD, "PAD весов не совпадает с prepared"
    assert model.config.n_positions >= histories.shape[1]
    # Если рядом с весами есть mapping, проверяем семантику индексов, а не только размер словаря.
    nearby_mapping = (path if path.is_dir() else path.parent) / "item_mapping.json"
    if nearby_mapping.exists():
        assert read_json(nearby_mapping)["item_id_to_index"] == item_to_index
    else:
        print("Mapping рядом с весами отсутствует: убедитесь, что веса обучались с item_mapping.json из этого prepared.")
    return model.float().to(DEVICE).eval()

model = load_checkpoint(CHECKPOINT_PATH)
print(f"Загружено {sum(p.numel() for p in model.parameters()):,} параметров; FP32; {DEVICE}")


In [ ]:
def stable_topk(scores, k=20):
    # Входные столбцы уже идут по item_index; стабильная сортировка сохраняет порядок при ties.
    if not torch.isfinite(scores).all():
        raise ValueError("В scores есть NaN/Inf")
    return torch.argsort(scores, dim=-1, descending=True, stable=True)[:, :k]

def per_user_metrics(recs, truth, seen):
    out = {}
    for k in [10, 20]:
        hits = recs[:, :k] == np.asarray(truth)[:, None]
        out[f"Recall@{k}"] = hits.any(axis=1).astype(float)
        out[f"NDCG@{k}"] = (hits / np.log2(np.arange(k) + 2)).sum(axis=1)
        out[f"SeenRepeatRate@{k}"] = seen[:, :k].mean(axis=1)
        out[f"UsersWithSeenRepeat@{k}"] = seen[:, :k].any(axis=1).astype(float)
    return out

# Контрольные примеры с известными ответами: rank=1, rank=2, промах.
toy = np.tile(np.arange(20), (3, 1))
toy_seen = np.zeros((3,20), dtype=bool)
toy_seen[0, 0] = True
m = per_user_metrics(toy, np.array([0,1,25]), toy_seen)
assert np.allclose(m["Recall@10"], [1,1,0])
assert np.allclose(m["NDCG@20"], [1,1/np.log2(3),0])
assert np.isclose(m["SeenRepeatRate@10"].mean(), 1/30)
assert stable_topk(torch.zeros(1,25),20).tolist()[0] == list(range(20))
assert contains_sorted(np.array([0,4,8]), np.array([0,3,8,9])).tolist() == [True,False,True,False]
assert not contains_sorted(np.array([], dtype=np.int64), np.array([0])).any()
print("Контрольные тесты метрик и порядка ties пройдены.")


In [ ]:
# GPT-2 получает ID фильмов напрямую: текстовый GPT-2 tokenizer здесь не нужен.
# Проецируем в словарь только последнее непустое состояние, чтобы экономить память.
@torch.inference_mode()
def predict_top20(model, rows):
    lens = lengths[rows].astype(np.int64)
    width = int(lens.max())
    ids = torch.as_tensor(histories[rows, :width].astype(np.int64), device=DEVICE)
    ln = torch.as_tensor(lens, device=DEVICE)
    attention_mask = (torch.arange(width, device=DEVICE)[None, :] < ln[:, None]).long()
    hidden = model.transformer(input_ids=ids, attention_mask=attention_mask,
                               use_cache=False, return_dict=True).last_hidden_state
    last = hidden[torch.arange(len(rows), device=DEVICE), ln-1]
    scores = model.lm_head(last)[:, :N_ITEMS]
    return stable_topk(scores).cpu().numpy()

# Сверяем экономный forward с обычным LM forward на нескольких реальных контекстах.
with torch.inference_mode():
    rows = np.arange(min(3,N))
    ln = torch.as_tensor(lengths[rows].astype(np.int64), device=DEVICE)
    width = int(ln.max())
    ids = torch.as_tensor(histories[rows,:width].astype(np.int64), device=DEVICE)
    am = (torch.arange(width,device=DEVICE)[None,:] < ln[:,None]).long()
    logits = model(input_ids=ids, attention_mask=am, use_cache=False).logits
    full_scores = logits[torch.arange(len(rows),device=DEVICE),ln-1,:N_ITEMS]
    hidden = model.transformer(input_ids=ids,attention_mask=am,use_cache=False).last_hidden_state
    compact_scores = model.lm_head(hidden[torch.arange(len(rows),device=DEVICE),ln-1])[:,:N_ITEMS]
    torch.testing.assert_close(compact_scores, full_scores, rtol=1e-5, atol=1e-5)
    del logits, hidden, compact_scores, full_scores

if OUTPUT_DIR.exists():
    raise FileExistsError(f"{OUTPUT_DIR} уже существует. Задайте новую OUTPUT_DIR, чтобы сохранить baseline.")
top20 = np.empty((N,20), dtype=np.int32)
t0 = time.perf_counter()
for start in tqdm(range(0,N,BATCH_SIZE), desc=f"GPT-2 · {SPLIT}"):
    rows = np.arange(start, min(start+BATCH_SIZE,N))
    top20[rows] = predict_top20(model, rows)
elapsed = time.perf_counter()-t0
assert ((top20>=0)&(top20<N_ITEMS)).all()
assert (np.diff(np.sort(top20,axis=1),axis=1)>0).all()
seen = seen_flags(top20, np.arange(N))
per_user = pd.DataFrame({"user_id":users, "target_index":targets, "target_seen":target_seen,
                         **per_user_metrics(top20, targets, seen)})
metric_columns = [c for c in per_user if "@" in c]
summary = {c:float(per_user[c].mean()) for c in metric_columns}
summary["Coverage@10"] = float(np.unique(top20[:,:10]).size/N_ITEMS)
summary["Coverage@20"] = float(np.unique(top20).size/N_ITEMS)
summary["TargetSeenRate"] = float(target_seen.mean())
display(pd.DataFrame.from_dict(summary,orient="index",columns=["value"]).style.format("{:.6f}"))
print(f"Полная оценка: {N:,} пользователей; {elapsed/60:.1f} мин.")


In [ ]:
# Фиксируем входы и протокол, чтобы последующее сравнение обнаруживало смену данных/когорты.
data_hashes = {name:sha256(DATA_DIR/name) for name in required}
cp = Path(CHECKPOINT_PATH)
weight_paths = ([cp] if cp.is_file() else sorted(p for p in cp.rglob("*")
                if p.is_file() and (p.suffix in {".safetensors", ".bin", ".pt", ".pth"}
                                   or p.name in {"config.json", "item_mapping.json"})))
assert weight_paths, "Не найдены файлы весов для fingerprint"
weight_hashes = {(p.name if cp.is_file() else str(p.relative_to(cp))):sha256(p) for p in weight_paths}
contract = {"protocol_version":"gpt2_next_item_v1", "split":SPLIT, "n_users":N,
            "catalog_size":N_ITEMS, "data_sha256":data_hashes,
            "candidates":"all_train_items", "filter_seen":False,
            "tie_break":"item_index_ascending", "precision":"float32",
            "history_for_repeats":"full_train_plus_validation" if SPLIT=="test" else "full_train",
            "context_width":int(histories.shape[1]),
            "aggregation":"macro_users_one_target", "coverage_denominator":N_ITEMS}
manifest = {"created_utc":datetime.now(timezone.utc).isoformat(),
            "contract":contract, "checkpoint_path":str(cp.resolve()), "weights_sha256":weight_hashes,
            "model_config":model.config.to_dict(), "environment":environment,
            "evaluation_seconds":elapsed, "batch_size":BATCH_SIZE,
            "cohort_audit":stats["splits"][SPLIT]["evaluation"]}

OUTPUT_DIR.mkdir(parents=True,exist_ok=False)
def save_json(path, obj):
    Path(path).write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding="utf-8")
save_json(OUTPUT_DIR/"metrics.json",summary)
save_json(OUTPUT_DIR/"manifest.json",manifest)
save_json(OUTPUT_DIR/"cohort_audit.json",stats["splits"][SPLIT]["evaluation"])
pd.Series(summary,name="value").rename_axis("metric").to_csv(OUTPUT_DIR/"metrics.csv")
per_user.to_csv(OUTPUT_DIR/"per_user_metrics.csv",index=False)
np.savez_compressed(OUTPUT_DIR/"recommendations.npz", users=users, targets=targets,
                    top20_indices=top20, top20_item_ids=index_to_item[top20], seen_flags=seen)
save_json(OUTPUT_DIR/"item_mapping.json",mapping)
print("Сохранено:", OUTPUT_DIR.resolve())
print("metrics.json/csv, manifest.json, cohort_audit.json, per_user_metrics.csv, recommendations.npz, item_mapping.json")


## Сравнение после RL

Сохраните папку baseline. После RL запустите копию этого ноутбука с новыми весами,
новой `OUTPUT_DIR` и `REFERENCE_DIR`, указывающей на папку baseline.
Данные, mapping, аудитория, кандидаты и формулы должны совпадать. Таблица покажет абсолютную
разницу; для Recall/NDCG/Coverage рост означает увеличение метрики, для повторов снижение
означает меньше рекомендаций уже просмотренных фильмов. Рост Coverage не означает автоматически
рост качества. Разница сама по себе не является проверкой статистической значимости.
Промежуточные решения RL принимайте на validation, не по test.


In [ ]:
if REFERENCE_DIR is not None:
    ref = Path(REFERENCE_DIR)
    old_manifest = read_json(ref/"manifest.json")
    if old_manifest["contract"] != contract:
        raise ValueError("Протокол или данные изменились: прямое сравнение с baseline запрещено.")
    old = read_json(ref/"metrics.json")
    comparison = pd.DataFrame({"baseline":old, "current":summary})
    comparison["delta_current_minus_baseline"] = comparison["current"]-comparison["baseline"]
    comparison.to_csv(OUTPUT_DIR/"comparison_vs_baseline.csv",index_label="metric")
    display(comparison.style.format("{:.6f}"))
else:
    print("Исходная точка сохранена. Для сравнения после RL укажите REFERENCE_DIR.")
